# Teaching Micrograd: Backpropagation from Scratch

This notebook walks through Andrej Karpathy's [micrograd](https://github.com/karpathy/micrograd) library to teach how neural network training actually works under the hood. We build everything from first principles — no assumed math background beyond high-school algebra.

---
## 1. What is Autograd and Why Do We Need It?

When we train a neural network we are doing one thing over and over:

> **Make a prediction → measure how wrong it is → nudge every weight a little in the direction that makes it less wrong → repeat.**

The "nudge" step requires knowing **the gradient of the loss with respect to every weight** — i.e., "if I increase this weight by a tiny bit, does the loss go up or down, and by how much?"

A network can have millions of weights. Computing those gradients by hand for every possible architecture would be impossibly tedious. **Automatic differentiation (autograd)** is the technique that does it for us automatically, no matter how the computation is structured.

Micrograd is a tiny (~100-line) autograd engine that does exactly this. Understanding it gives you the mental model behind PyTorch, TensorFlow, and JAX.

---
## 2. Build the `Value` Class Step by Step

The heart of micrograd is a class called `Value`. It wraps a single number but also:

1. **Remembers which operation created it** (add, multiply, …)
2. **Remembers its inputs** ("children" in the computation graph)
3. **Stores its gradient** `.grad` (how much the final output changes if *this* value changes)

We'll build it in stages so each piece is clear.

### Stage 1 — just hold data and track the graph

In [ ]:
class Value:
    """A scalar value that knows how it was computed."""

    def __init__(self, data, _children=(), _op=''):
        self.data = data          # the actual number
        self.grad = 0.0           # gradient starts at zero ("don't know yet")
        self._prev = set(_children)  # the Value nodes that fed into this one
        self._op = _op            # the operation that created this node

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# Let's create two values and manually build a graph
a = Value(2.0)
b = Value(3.0)
print(a)
print(b)

### Stage 2 — add arithmetic operations

We overload Python's `+` and `*` operators so that `Value + Value` returns a new `Value` that remembers its parents. This is how the **computation graph** (also called the *expression graph*) gets built automatically just by writing normal Python math.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        # If 'other' is a plain Python number, wrap it in a Value first
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        return out

    # These let Python evaluate  2 + Value(...)  and  3 * Value(...)  correctly
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


a = Value(2.0)
b = Value(3.0)
c = a + b      # c.data == 5.0, c._prev == {a, b}, c._op == '+'
d = a * b      # d.data == 6.0
e = c + d      # e.data == 11.0

print(f"c = a + b = {c}")
print(f"d = a * b = {d}")
print(f"e = c + d = {e}")
print(f"e was created by op: '{e._op}'")
print(f"e's parents: {e._prev}")

---
## 3. Derivative Intuition — The Limit Definition

Now that we have a Value graph, we need to answer: **if I nudge one value slightly, how much does the final output change?** That's what a derivative measures.

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

In plain English: nudge $x$ by a tiny $h$, observe how much $f(x)$ moves, take the ratio. Let's visualise it with $f(x) = x^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return x ** 2

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

x0 = 3.0
h_values = [1.0, 0.5, 0.1, 0.01, 0.001, 0.0001]

print("Limit definition of derivative of x^2 at x=3 (true answer = 6):")
for h in h_values:
    approx = numerical_derivative(f, x0, h)
    print(f"  h={h:.4f}  =>  approx derivative = {approx:.6f}")

xs = np.linspace(0, 5, 200)
h_show = 1.0

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, f(xs), label=r'$f(x)=x^2$', linewidth=2)

true_slope = 2 * x0
tangent = f(x0) + true_slope * (xs - x0)
ax.plot(xs, tangent, '--', label=f'Tangent (slope={true_slope})', color='green')

secant_slope = numerical_derivative(f, x0, h_show)
secant = f(x0) + secant_slope * (xs - x0)
ax.plot(xs, secant, ':', label=f'Secant h={h_show} (slope={secant_slope:.2f})', color='red')

ax.scatter([x0, x0 + h_show], [f(x0), f(x0 + h_show)], color='red', zorder=5)
ax.set_ylim(-2, 30); ax.set_xlim(0, 5)
ax.legend(); ax.set_title('Derivative as limit of secant slope')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
plt.tight_layout(); plt.show()

print("\nAs h → 0, the secant slope → the tangent slope (the true derivative).")

---
## 4. Manual Backprop on a Single Neuron (tanh activation)

Before we automate gradients, let's compute them by hand for one neuron so we know what we're aiming for.

A single neuron computes:
$$o = \tanh\!\left(\sum_i w_i x_i + b\right)$$

The **chain rule** says: to find how the final output changes with respect to any intermediate value, multiply the local derivative at each step together as you walk backward through the graph.

We'll trace through this by hand and then verify with numerical derivatives.

In [ ]:
import math

# Inputs
x1 = Value(2.0)   # input 1
x2 = Value(0.0)   # input 2
# Weights
w1 = Value(-3.0)
w2 = Value(1.0)
# Bias
b  = Value(6.8813735870195432)  # chosen so tanh(n) comes out to a nice number

# --- Forward pass ---
x1w1 = x1 * w1          # -6.0
x2w2 = x2 * w2          # 0.0
n    = x1w1 + x2w2 + b  # pre-activation sum ≈ 0.8814
# tanh: squashes the sum to (-1, 1)
o    = Value(math.tanh(n.data), (n,), 'tanh')

print(f"n (pre-activation) = {n.data:.4f}")
print(f"o = tanh(n)        = {o.data:.4f}")

# --- Manual backward pass ---
# Start: gradient of output w.r.t. itself = 1
o.grad = 1.0

# Derivative of tanh: d/dx tanh(x) = 1 - tanh(x)^2
# So dn/do = 1 - o^2  (chain rule: multiply by o.grad = 1)
n.grad = (1 - o.data**2) * o.grad
print(f"\nn.grad = {n.grad:.4f}  (from tanh derivative)")

# n = x1w1 + x2w2 + b  → addition just passes the gradient through unchanged
x1w1.grad = n.grad
x2w2.grad = n.grad
b.grad    = n.grad

# x1w1 = x1 * w1  → multiply: grad flows as  local_input * upstream_grad
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

print("\nManually computed gradients:")
print(f"  x1.grad = {x1.grad:.4f}")
print(f"  w1.grad = {w1.grad:.4f}")
print(f"  x2.grad = {x2.grad:.4f}")
print(f"  w2.grad = {w2.grad:.4f}")
print(f"  b.grad  = {b.grad:.4f}")

# --- Verify with numerical derivatives ---
def verify_numerical(val, other_vals, forward_fn, h=1e-5):
    orig = val.data
    val.data = orig + h
    plus  = forward_fn()
    val.data = orig - h
    minus = forward_fn()
    val.data = orig
    return (plus - minus) / (2 * h)

def forward():
    n_val = x1.data*w1.data + x2.data*w2.data + b.data
    return math.tanh(n_val)

print("\nNumerical verification:")
for name, val in [("x1", x1), ("w1", w1), ("x2", x2), ("w2", w2), ("b", b)]:
    num_grad = verify_numerical(val, [], forward)
    print(f"  {name}: manual={val.grad:.4f}  numerical={num_grad:.4f}  match={abs(val.grad - num_grad) < 1e-4}")

---
## 5. Automate Backprop — Add `_backward` Closures

Doing the backward pass by hand every time would be tedious and error-prone. The key insight is:

> **Each operation knows its own local derivative.** We can bake the gradient computation into a `_backward` function right when the operation is performed.

Then the backward pass just calls `_backward()` on every node in reverse order — the chain rule is applied automatically.

Here is the full `Value` class with `_backward` closures for every operation:

In [ ]:
import math

class Value:
    """Scalar value with automatic differentiation."""

    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None   # default: leaf node, nothing to propagate
        self._prev = set(_children)
        self._op = _op

    # ---------- forward operations ----------

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # Gradient of a sum just passes through unchanged to both inputs
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # d(a*b)/da = b  and  d(a*b)/db = a
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')

        def _backward():
            # Power rule: d(x^n)/dx = n * x^(n-1)
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')

        def _backward():
            # d/dx tanh(x) = 1 - tanh(x)^2
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')

        def _backward():
            # Gradient is 1 if input > 0, else 0
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    # ---------- convenience methods ----------

    def __neg__(self):          return self * -1
    def __radd__(self, other):  return self + other
    def __sub__(self, other):   return self + (-other)
    def __rsub__(self, other):  return other + (-self)
    def __rmul__(self, other):  return self * other
    def __truediv__(self, other):  return self * other ** -1
    def __rtruediv__(self, other): return other * self ** -1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"


# Quick smoke test: d/dx (x^2) at x=3 should be 6
x = Value(3.0)
y = x ** 2
y._backward()      # we'll call backward() properly in the next section
# But first set the seed gradient manually
y.grad = 1.0
y._backward()
print(f"x={x.data}, y=x^2={y.data}, dy/dx={x.grad}  (expected 6.0)")

---
## 5b. `exp()` as a Primitive — and Decomposing `tanh`

We implemented `tanh` as a single operation above. But `tanh` is not magic — it's built from more basic pieces:

$$\tanh(x) = \frac{e^{2x} - 1}{e^{2x} + 1}$$

If we add `exp()` as a primitive, we can build `tanh` ourselves using only `exp`, `+`, `-`, and `/`. The chain rule doesn't care *how* you slice up the computation — it will produce the same gradients either way.

This is a crucial insight: **the backward pass is correct regardless of how finely or coarsely you decompose an operation**, as long as each primitive's `_backward` is right.

---
## 6. Topological Sort and Calling `loss.backward()`

We need to call `_backward()` on every node in the right order: **outputs before inputs** (reverse topological order).

A **topological sort** visits each node only after all the nodes that depend on it have been visited. We build this order with a depth-first search, then reverse it.

The `.backward()` method does this automatically.

In [ ]:
# Add .backward() to the Value class  (we patch it onto the class we already defined)

def backward(self):
    # Build a list of all nodes in topological order
    topo = []
    visited = set()

    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:   # recurse into inputs first
                build_topo(child)
            topo.append(v)          # append AFTER children → reverse post-order

    build_topo(self)

    # Seed: gradient of the output w.r.t. itself is always 1
    self.grad = 1.0
    # Walk backward and call each node's _backward closure
    for v in reversed(topo):
        v._backward()

Value.backward = backward   # attach to the class


# --- Demonstrate on the single-neuron example from section 4 ---
x1 = Value(2.0);  x2 = Value(0.0)
w1 = Value(-3.0); w2 = Value(1.0)
b  = Value(6.8813735870195432)

n = x1*w1 + x2*w2 + b
o = n.tanh()          # output of the neuron

o.backward()          # <-- one call does everything!

print("Automated backprop results:")
for name, val in [("x1",x1),("w1",w1),("x2",x2),("w2",w2),("b",b)]:
    print(f"  {name}.grad = {val.grad:.4f}")

# These should match the manually computed values from section 4

---
## 7. Build Neuron → Layer → MLP

Now we can stack our `Value` engine into actual neural network building blocks.

| Class | What it does |
|---|---|
| `Neuron` | One dot product + bias + activation |
| `Layer` | Several neurons in parallel (same inputs, different weights) |
| `MLP` | Several layers in sequence (multi-layer perceptron) |

In [ ]:
import random

class Module:
    """Base class: knows how to zero gradients and list parameters."""
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    def parameters(self):
        return []


class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        # Random weights in [-1, 1], bias starts at 0
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin   # whether to apply ReLU activation

    def __call__(self, x):
        # Dot product: sum(wi * xi) + bias
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"


class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out   # unwrap single-output layers

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


class MLP(Module):
    def __init__(self, nin, nouts):
        # nouts is a list of layer sizes, e.g. [4, 4, 1]
        # The last layer is linear (no activation) — common for regression / binary output
        sizes = [nin] + nouts
        self.layers = [
            Layer(sizes[i], sizes[i+1], nonlin=(i != len(nouts)-1))
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


random.seed(42)
model = MLP(nin=3, nouts=[4, 4, 1])   # 3 inputs → two hidden layers of 4 → 1 output
print(model)
print(f"Total parameters: {len(model.parameters())}")

# Forward pass on a single sample
sample = [Value(1.0), Value(2.0), Value(3.0)]
out = model(sample)
print(f"Output for sample [1,2,3]: {out}")

---
## 8. Training Loop — Dataset, Loss, Gradient Descent

We now have everything we need to train a neural network:

1. **Forward pass** — run inputs through the network
2. **Compute loss** — mean squared error between predictions and targets
3. **Backward pass** — `loss.backward()` fills in `.grad` for every parameter
4. **Update** — nudge each parameter in the direction that lowers the loss
5. **Zero gradients** — reset `.grad` to 0 before the next iteration (otherwise they accumulate)

We'll use a tiny 4-sample dataset — the classic XOR-like binary classification problem from Karpathy's demo.

In [ ]:
random.seed(42)

# ---- Dataset ----
# 4 data points, 3 features each
X = [
    [2.0,  3.0, -1.0],
    [3.0, -1.0,  0.5],
    [0.5,  1.0,  1.0],
    [1.0,  1.0, -1.0],
]
# Binary labels: +1 or -1
y_true = [1.0, -1.0, -1.0, 1.0]

# ---- Model ----
model = MLP(nin=3, nouts=[4, 4, 1])

learning_rate = 0.05
losses = []   # track loss over time

for step in range(100):

    # 1. Forward pass: get predictions for all samples
    y_pred = [model(x) for x in X]

    # 2. Compute mean squared error loss
    #    Loss = mean( (y_pred - y_true)^2 )
    loss = sum((pred - gt)**2 for pred, gt in zip(y_pred, y_true)) * (1.0 / len(y_true))

    # 3. Zero gradients from the previous step
    model.zero_grad()

    # 4. Backward pass: compute all gradients
    loss.backward()

    # 5. Gradient descent: move each parameter opposite to its gradient
    for p in model.parameters():
        p.data -= learning_rate * p.grad

    losses.append(loss.data)

    if step % 10 == 0 or step == 99:
        preds = [round(p.data) for p in y_pred]
        print(f"step {step:3d}  loss={loss.data:.4f}  preds={[f'{p.data:.2f}' for p in y_pred]}")

print("\nFinal predictions vs targets:")
for pred, target in zip(y_pred, y_true):
    sign = '+1' if pred.data > 0 else '-1'
    print(f"  pred={pred.data:+.3f} ({sign})  target={target:+.0f}  correct={sign == ('+1' if target>0 else '-1')}")

### Loss curve

A healthy training run shows the loss decreasing smoothly over time. Let's plot it.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, linewidth=2, color='steelblue')
plt.xlabel('Training step')
plt.ylabel('MSE Loss')
plt.title('Micrograd MLP Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss went from {losses[0]:.4f} → {losses[-1]:.4f}")

---
## 9. Quick PyTorch Comparison — They Agree!

Let's verify that micrograd's gradients match PyTorch's gradients on the same computation. This is the same test used in the micrograd test suite.

In [ ]:
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed — skipping comparison (install with: pip install torch)")

if TORCH_AVAILABLE:
    # ---- Run with micrograd ----
    a_mg = Value(-4.0)
    b_mg = Value(2.0)
    c = a_mg + b_mg
    d = a_mg * b_mg + b_mg**3
    c = c + c + 1
    c = c + 1 + c + (-a_mg)
    d = d + d * 2 + (b_mg + a_mg).relu()
    d = d + 3 * d + (b_mg - a_mg).relu()
    e = c - d
    f = e**2
    g_mg = f / 2.0
    g_mg = g_mg + 10.0 / f
    g_mg.backward()

    # ---- Run the identical computation with PyTorch ----
    a_pt = torch.tensor([-4.0], dtype=torch.float64, requires_grad=True)
    b_pt = torch.tensor([ 2.0], dtype=torch.float64, requires_grad=True)
    c = a_pt + b_pt
    d = a_pt * b_pt + b_pt**3
    c = c + c + 1
    c = c + 1 + c + (-a_pt)
    d = d + d * 2 + (b_pt + a_pt).relu()
    d = d + 3 * d + (b_pt - a_pt).relu()
    e = c - d
    f = e**2
    g_pt = f / 2.0
    g_pt = g_pt + 10.0 / f
    g_pt.backward()

    tol = 1e-6
    fwd_match = abs(g_mg.data - g_pt.item()) < tol
    a_match   = abs(a_mg.grad - a_pt.grad.item()) < tol
    b_match   = abs(b_mg.grad - b_pt.grad.item()) < tol

    print("Forward pass:")
    print(f"  micrograd g = {g_mg.data:.6f}")
    print(f"  pytorch   g = {g_pt.item():.6f}")
    print(f"  match: {fwd_match}")

    print("\nBackward pass (gradients):")
    print(f"  a.grad:  micrograd={a_mg.grad:.6f}  pytorch={a_pt.grad.item():.6f}  match={a_match}")
    print(f"  b.grad:  micrograd={b_mg.grad:.6f}  pytorch={b_pt.grad.item():.6f}  match={b_match}")

    if fwd_match and a_match and b_match:
        print("\n✓ micrograd and PyTorch agree to within", tol)

---
## Recap: What We Built

| Concept | Where we saw it |
|---|---|
| Computation graph | Section 2 — `Value._prev` tracks parents |
| Derivative as a limit | Section 3 — plotted secant converging to tangent |
| Chain rule by hand | Section 4 — manual backprop through a neuron |
| `_backward` closures | Section 5 — each op bakes in its own local gradient |
| Topological sort | Section 6 — guarantees correct backward order |
| Neural network layers | Section 7 — `Neuron`, `Layer`, `MLP` on top of `Value` |
| Training loop | Section 8 — loss drops, predictions flip to correct sign |
| Equivalence to PyTorch | Section 9 — same numbers, same gradients |

The whole engine — forward pass, backward pass, graph construction — is fewer than 100 lines of pure Python. PyTorch, TensorFlow, and JAX do the same thing, just extended to tensors (arrays of numbers) and compiled to run on GPUs.